# CW305 MNIST Neural Network Classifier — Power Trace Capture

This notebook implements a fully connected neural network (784-64-64-32-10) on the CW305 FPGA board for MNIST digit classification.
It demonstrates:
1. Loading MNIST data from a CSV file.
2. Transferring image data to the FPGA via USB registers.
3. Triggering the NN classification and capturing power traces.
4. Reading the classification result and comparing it with the label.

In [ ]:
import chipwhisperer as cw
import numpy as np
import matplotlib.pyplot as plt
import time
import pandas as pd

BITFILE      = 'test2.runs/impl_1/cw305_top.bit'
DEFINES_FILE = 'test2.srcs/sources_1/new/cw305_user_defines.v'
DATASET_FILE = 'MNIST-10000-784.csv'

print(f'ChipWhisperer version: {cw.__version__}')

## 1 — Connect to Scope and Target

In [ ]:
scope = cw.scope()
scope.default_setup()

class MyCW305(cw.targets.CW305):
    def __init__(self):
        super().__init__()
        self.registers      = 5   # Number of `define REG_NN_* in defines file
        self.bytecount_size = 7

target = cw.target(scope, MyCW305,
                   bsfile        = BITFILE,
                   force         = True,
                   fpga_id       = '100t',
                   platform      = 'cw305',
                   defines_files = [DEFINES_FILE])

print('FPGA Programmed and Registers Parsed.')

## 2 — Configure Clock and Capture

In [ ]:
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(False, 0)
target.pll.pll_outenable_set(True,  1)
target.pll.pll_outenable_set(False, 2)
target.pll.pll_outfreq_set(10e6, 1)

scope.io.hs2 = 'disabled'
scope.clock.adc_src = 'extclk_x4'
scope.clock.reset_adc()
time.sleep(0.2)
assert scope.clock.adc_locked, 'ADC failed to lock to FPGA clock.'

scope.adc.samples = 10000 # NN takes time
scope.adc.offset = 0
scope.trigger.triggers = 'tio4'
print(f'ADC locked. Samples: {scope.adc.samples}')

## 3 — Load MNIST Dataset

In [ ]:
df = pd.read_csv(DATASET_FILE)
labels = df['label'].values
pixels = df.drop('label', axis=1).values

def get_binary_image(index):
    img = pixels[index]
    img_bin = (img > 127).astype(np.uint8)
    return img_bin, labels[index]

img, label = get_binary_image(0)
plt.imshow(img.reshape(28,28), cmap='gray')
plt.title(f'Label: {label}')
plt.show()

## 4 — Classification and Trace Capture

In [ ]:
def classify_and_capture(img_bin):
    # 1. Write image to FPGA memory
    # Addresses 0 to 783 are mapped to image memory
    for i, val in enumerate(img_bin):
        target.write(i, [val])
    
    # 2. Arm scope
    scope.arm()
    
    # 3. Trigger NN
    target.write(target.REG_NN_GO, [0x01])
    
    # 4. Wait for done
    timeout = 100
    while timeout > 0:
        if target.read(target.REG_NN_DONE, 1)[0] == 1:
            break
        time.sleep(0.01)
        timeout -= 1
    
    # 5. Capture trace
    ret = scope.capture()
    if ret:
        print('Capture timeout!')
        return None, None
    
    trace = scope.get_last_trace()
    
    # 6. Read result
    result = target.read(target.REG_NN_RESULT, 1)[0]
    
    return result, trace

idx = 0
img, label = get_binary_image(idx)
res, trace = classify_and_capture(img)

print(f'Image {idx} - Actual: {label}, Predicted: {res}')
plt.plot(trace)
plt.title('Power Trace during MNIST Classification')
plt.show()

## 5 — Batch Capture

Capture 10 traces and see the accuracy.

In [ ]:
traces = []
results = []
N = 10

for i in range(N):
    img, label = get_binary_image(i)
    res, trace = classify_and_capture(img)
    results.append((label, res))
    traces.append(trace)
    print(f'Done {i+1}/{N}')

correct = sum(1 for l, r in results if l == r)
print(f'Accuracy on {N} samples: {correct/N*100}%')